# Oregon PM2.5 Air Quality Analysis
**Author:** Lexie Smith  
**Data Source:** EPA Air Quality System (AQS) — Daily PM2.5 FRM/FEM Mass (Parameter 88101)  
**Data Range:** 1999–2022 (Oregon monitoring stations only)  
**Tools:** Python, MySQL, Power BI

## Project Overview
This notebook documents the data pipeline for an analysis of PM2.5 air quality trends across Oregon counties from 1999 to 2022. The pipeline covers:
1. Automated bulk download of 35 years of EPA AQS daily PM2.5 data
2. File extraction and organization
3. Data exploration and column selection
4. Loading filtered Oregon data into a normalized MySQL database

SQL analysis queries and Power BI dashboard are maintained separately (see repository).

## Research Questions
- Has the number of days exceeding the federal PM2.5 standard (35 µg/m³) changed over time in Oregon?
- Which counties show the highest PM2.5 exposure when normalized by monitoring station count?
- Are unhealthy air days concentrated in specific seasons, and what does that suggest about pollution sources?

## Step 1: Download EPA AQS Daily PM2.5 Data

The EPA Air Quality System (AQS) publishes annual CSV files for each criteria pollutant at:
https://aqs.epa.gov/aqsweb/airdata/download_files.html

Parameter code `88101` corresponds to PM2.5 FRM/FEM Mass, the regulatory standard measurement.  
Files are downloaded programmatically for all available years (1990–2024) and saved as zip archives.

**Note:** Early years (1990–1998) contain no Oregon data. PM2.5 became a regulated pollutant under the NAAQS in 1997, so widespread state monitoring did not begin until 1999.

In [ ]:
import requests
import os

# Set your local data directory
save_folder = os.path.join(os.path.expanduser("~"), "Documents", "DEQ_project", "data", "pm25_daily")
os.makedirs(save_folder, exist_ok=True)

# EPA AQS bulk download URL pattern — parameter 88101 = PM2.5 FRM/FEM Mass
base_url = "https://aqs.epa.gov/aqsweb/airdata/daily_88101_{year}.zip"

years = range(1990, 2025)

for year in years:
    url = base_url.format(year=year)
    filename = os.path.join(save_folder, f"daily_pm25_{year}.zip")
    
    print(f"Downloading {year}...")
    response = requests.get(url)
    
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"  Saved {year}")
    else:
        print(f"  Skipped {year} (not available)")

print("Download complete.")

## Step 2: Extract and Organize Files

Downloaded zip files are extracted to a separate folder.  
Some years (2020–2023) unzipped into subfolders rather than directly — these are moved to the flat directory structure for consistent processing.

In [ ]:
import zipfile
import shutil

unzip_folder = os.path.join(os.path.expanduser("~"), "Documents", "DEQ_project", "data", "pm25_unzipped")
os.makedirs(unzip_folder, exist_ok=True)

# Extract all zip files
for filename in os.listdir(save_folder):
    if filename.endswith(".zip"):
        zip_path = os.path.join(save_folder, filename)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(unzip_folder)
        print(f"Extracted {filename}")

# Some years extract into subfolders — flatten to consistent structure
for item in os.listdir(unzip_folder):
    item_path = os.path.join(unzip_folder, item)
    if os.path.isdir(item_path):
        csv_name = item + ".csv"
        src = os.path.join(item_path, csv_name)
        dst = os.path.join(unzip_folder, csv_name)
        if os.path.exists(src):
            shutil.move(src, dst)
            shutil.rmtree(item_path)
            print(f"Flattened {item}")

# Confirm all files present
csv_files = [f for f in os.listdir(unzip_folder) if f.endswith(".csv")]
print(f"\nTotal CSV files ready: {len(csv_files)}")

## Step 3: Data Exploration

Before loading into the database, the raw data structure is examined to inform schema design and column selection.

**Key decisions made during exploration:**
- Filter to `State Name == 'Oregon'` — files contain national data
- Filter to `Sample Duration == '24 HOUR'` — excludes 1-hour and block-average readings to ensure consistent daily measurements
- Retain columns relevant to geographic identification, measurement values, and data quality flags
- `Event Type` column retained — 'Excluded' values often indicate wildfire-impacted days flagged under EPA exceptional events policy
- `Arithmetic Mean` is the primary PM2.5 metric; the federal 24-hour standard threshold is 35 µg/m³

In [ ]:
import pandas as pd

# Inspect a sample year to understand structure
sample_file = os.path.join(unzip_folder, "daily_88101_2020.csv")
df_sample = pd.read_csv(sample_file)

print(f"Shape: {df_sample.shape}")
print(f"\nColumns: {df_sample.columns.tolist()}")
print(f"\nEvent Type values: {df_sample['Event Type'].unique()}")
print(f"Sample Duration values: {df_sample['Sample Duration'].unique()}")
print(f"\nOregon records in 2020: {len(df_sample[df_sample['State Name'] == 'Oregon'])}")

## Step 4: Load Data into MySQL

Data is loaded into a normalized MySQL database (`oregon_air_quality`) with two tables:
- `monitors` — one row per monitoring station (location, county, coordinates)
- `daily_readings` — one row per day per monitor (PM2.5 values, AQI, event type)

**Database credentials are stored as environment variables and not hardcoded.**  
Set the following environment variables before running:
- `MYSQL_USER`
- `MYSQL_PASSWORD`

**Data quality handling:**
- Missing values (`NaN`) are converted to `NULL` before insertion
- Date formats vary across years — a parser handles both `YYYY-MM-DD` and `M/D/YYYY` formats
- Duplicate monitor entries are skipped using `INSERT IGNORE`

In [ ]:
import os
import pandas as pd
import mysql.connector
from datetime import datetime

# Set your data folder path
unzip_folder = os.path.join(os.path.expanduser("~"), "Documents", "DEQ_project", "data", "pm25_unzipped")
# Load credentials from environment variables
MYSQL_USER = os.environ.get("MYSQL_USER", "root")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "")

def clean(val):
    """Convert NaN/None values to MySQL-compatible NULL."""
    if pd.isna(val):
        return None
    return val

def clean_date(val):
    """Parse dates handling both YYYY-MM-DD and M/D/YYYY formats across years."""
    if pd.isna(val):
        return None
    try:
        return datetime.strptime(str(val), "%Y-%m-%d").date()
    except ValueError:
        return datetime.strptime(str(val), "%m/%d/%Y").date()

conn = mysql.connector.connect(
    host="localhost",
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database="oregon_air_quality",
    connection_timeout=300
)
cursor = conn.cursor()

inserted_monitors = set()

for filename in sorted(os.listdir(unzip_folder)):
    if not filename.endswith(".csv"):
        continue

    year = filename.split("_")[2].split(".")[0]
    print(f"Processing {year}...")

    df = pd.read_csv(os.path.join(unzip_folder, filename), low_memory=False)

    # Filter to Oregon 24-hour readings only
    df = df[df["State Name"] == "Oregon"]
    df = df[df["Sample Duration"] == "24 HOUR"]

    if df.empty:
        print(f"  No Oregon data for {year} — skipping")
        continue

    # Construct unique site identifier: state-county-site (e.g. 41-051-0001)
    df["site_id"] = (
        df["State Code"].astype(str).str.zfill(2) + "-" +
        df["County Code"].astype(str).str.zfill(3) + "-" +
        df["Site Num"].astype(str).str.zfill(4)
    )

    # Insert monitor metadata (skip duplicates)
    for _, row in df.drop_duplicates("site_id").iterrows():
        if row["site_id"] not in inserted_monitors:
            cursor.execute("""
                INSERT IGNORE INTO monitors
                (site_id, state_code, county_code, site_num,
                 state_name, county_name, city_name, latitude, longitude)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
            """, (
                clean(row["site_id"]), clean(row["State Code"]), clean(row["County Code"]),
                clean(row["Site Num"]), clean(row["State Name"]), clean(row["County Name"]),
                clean(row["City Name"]), clean(row["Latitude"]), clean(row["Longitude"])
            ))
            inserted_monitors.add(row["site_id"])

    # Insert daily readings
    for _, row in df.iterrows():
        cursor.execute("""
            INSERT INTO daily_readings
            (site_id, date_local, event_type, observation_count,
             observation_percent, arithmetic_mean, max_value, aqi)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            clean(row["site_id"]),
            clean_date(row["Date Local"]),
            clean(row["Event Type"]),
            clean(row["Observation Count"]),
            clean(row["Observation Percent"]),
            clean(row["Arithmetic Mean"]),
            clean(row["1st Max Value"]),
            clean(row["AQI"])
        ))

    conn.commit()
    print(f"  Done {year}")

print("\nAll data loaded successfully.")
cursor.close()
conn.close()

## Step 5: Verify Database Load

Final verification confirms expected row counts, date range, and year-by-year distribution.  
Note: 2023–2024 data is incomplete due to EPA certification lag — final analysis is limited to 1999–2022.

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database="oregon_air_quality",
    connection_timeout=300
)

summary = pd.read_sql("""
    SELECT 
        YEAR(date_local) AS year,
        COUNT(*) AS readings
    FROM daily_readings
    GROUP BY year
    ORDER BY year
""", conn)

total = pd.read_sql("SELECT COUNT(*) AS total FROM daily_readings", conn)
monitors = pd.read_sql("SELECT COUNT(*) AS total FROM monitors", conn)

print(f"Total readings: {total['total'].values[0]:,}")
print(f"Total monitoring stations: {monitors['total'].values[0]}")
print(f"\nReadings by year:")
print(summary.to_string(index=False))

conn.close()

In [ ]:
import pandas as pd
import os

unzip_folder = os.path.join(os.path.expanduser("~"), "Documents", "DEQ_project", "data", "pm25_unzipped")
df = pd.read_csv(os.path.join(unzip_folder, "daily_88101_1999.csv"))
df_oregon = df[df["State Name"] == "Oregon"]
print(df_oregon["Sample Duration"].value_counts())

## Next Steps

With data loaded into MySQL, analysis is performed using SQL queries in `oregon_air_quality_analysis.sql`.  
Results are visualized in the Power BI dashboard (see repository for `.pbix` file and screenshots).

**Key findings from SQL analysis:**
- 2020 had the most days exceeding the federal PM2.5 standard (35 µg/m³) in the 25-year record
- Southern Oregon counties (Klamath, Jackson) show the highest PM2.5 exposure per monitoring station
- September is the worst month for unhealthy air, consistent with Oregon's wildfire season
- January and December also show elevated unhealthy days, driven by winter temperature inversions trapping wood smoke in valley communities (Lane, Jackson, Klamath counties)

**Limitations:**
- County-level comparisons are affected by variation in monitoring station density
- Larger rural counties are underrepresented relative to their geographic area and population
- 2023–2024 data excluded due to EPA certification lag